# Verify Model Feature Names

Load the registered model from GCS and inspect the feature names it was trained on.
Use this to confirm the model expects canonical feature names (`sepal_length_cm`, etc.)
after retraining with the feature store pipeline.

## Config

In [9]:
PROJECT = "deeplearning-sahil"
REGION = "us-central1"
MODEL_NAME = "Iris-Classifier-XGBoost-staging"

## 1. Find the blessed model and its artifact URI

In [ ]:
from google.cloud import aiplatform_v1

client = aiplatform_v1.ModelServiceClient(
    client_options={"api_endpoint": f"{REGION}-aiplatform.googleapis.com"}
)

parent_models = list(client.list_models(
    request={
        "parent": f"projects/{PROJECT}/locations/{REGION}",
        "filter": f"display_name={MODEL_NAME}",
    }
))

model = client.get_model(name=parent_models[0].name + "@blessed")

print(f"Model: {model.display_name}")
print(f"Version: {model.version_id}")
print(f"Aliases: {list(model.version_aliases)}")
print(f"Created: {model.create_time}")
print(f"Artifact URI: {model.artifact_uri}")

## 2. Download and load model.joblib from GCS

In [7]:
import fsspec
import joblib

model_uri = model.artifact_uri.rstrip("/") + "/model.joblib"
print(f"Downloading: {model_uri}")

fs, _ = fsspec.core.url_to_fs(model_uri)
with fs.open(model_uri, "rb") as f:
    xgb_model = joblib.load(f)

print(f"Model type: {type(xgb_model).__name__}")

Downloading: gs://sb-vertex/staging/pipeline_root/57434141298/pipeline-iris-staging-20260613161458/choose-best-model_-2434796601957416960/best_model/model.joblib
Model type: RandomForestClassifier


/Users/shlba/Desktop/Docs/Study/code/Machine-learning-Ops-Deployment-Inference/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/shlba/Desktop/Docs/Study/code/Machine-learning-Ops-Deployment-Inference/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 3. Inspect feature names the model was trained on

In [8]:
print("Feature names from training (feature_names_in_):")
print(list(xgb_model.feature_names_in_))
print(f"\nNumber of features: {xgb_model.n_features_in_}")

expected = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]
actual = list(xgb_model.feature_names_in_)

if actual == expected:
    print("\n✅ Feature names match canonical names from feature platform")
else:
    print(f"\n❌ Feature name mismatch!")
    print(f"  Expected: {expected}")
    print(f"  Actual:   {actual}")

Feature names from training (feature_names_in_):
['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

Number of features: 4

❌ Feature name mismatch!
  Expected: ['sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm']
  Actual:   ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
